In [4]:
import mysql.connector
conn = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "sivakumar"
)
cursor = conn.cursor()
print("connection successful")

connection successful


In [2]:
cursor.execute("CREATE DATABASE IF NOT EXISTS cart2insights")

In [6]:
cursor.execute("USE cart2insights")

In [7]:
cursor.execute("SELECT DATABASE()")
print(cursor.fetchone())

('cart2insights',)


In [ ]:
# CUSTOMERS TABLE
create_customers = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id VARCHAR(50) PRIMARY KEY,
    customer_unique_id VARCHAR(50),
    customer_zip_code_prefix INT,
    customer_city VARCHAR(100),
    customer_state VARCHAR(10)
)
"""

cursor.execute(create_customers)

In [6]:
# PRODUCTS TABLE
create_products = """
CREATE TABLE IF NOT EXISTS products (
    product_id VARCHAR(50) PRIMARY KEY,
    product_category_name VARCHAR(100),
    product_name_length INT,
    product_description_length INT,
    product_photos_qty INT,
    product_weight_g INT,
    product_length_cm INT,
    product_height_cm INT,
    product_width_cm INT
)
"""

cursor.execute(create_products)

In [7]:
# SELLERS TABLE
create_sellers = """
CREATE TABLE IF NOT EXISTS sellers (
    seller_id VARCHAR(50) PRIMARY KEY,
    seller_zip_code_prefix INT,
    seller_city VARCHAR(100),
    seller_state VARCHAR(10)
)
"""

cursor.execute(create_sellers)

In [ ]:
# GEOLOCATION TABLE
create_geolocation = """
CREATE TABLE IF NOT EXISTS geolocation (
    geolocation_zip_code_prefix INT,
    geolocation_lat DECIMAL(10,7),
    geolocation_lng DECIMAL(10,7),
    geolocation_city VARCHAR(100),
    geolocation_state VARCHAR(10)
)
"""

cursor.execute(create_geolocation)

In [15]:
# CATEGORY TRANSLATION TABLE
create_translation = """
CREATE TABLE IF NOT EXISTS product_category_translation (
    product_category_name VARCHAR(100) PRIMARY KEY,
    product_category_name_english VARCHAR(100)
)
"""

cursor.execute(create_translation)

In [9]:
# ORDERS TABLE
create_orders = """
CREATE TABLE IF NOT EXISTS orders (
    order_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50),
    order_status VARCHAR(30),
    order_purchase_timestamp DATETIME,
    order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME,
    order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME,

    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
)
"""

cursor.execute(create_orders)

In [10]:
# ORDER ITEMS TABLE
create_order_items = """
CREATE TABLE IF NOT EXISTS order_items (
    order_id VARCHAR(50),
    order_item_id INT,
    product_id VARCHAR(50),
    seller_id VARCHAR(50),
    shipping_limit_date DATETIME,
    price DECIMAL(10,2),
    freight_value DECIMAL(10,2),

    PRIMARY KEY (order_id, order_item_id),

    FOREIGN KEY (order_id)
        REFERENCES orders(order_id),

    FOREIGN KEY (product_id)
        REFERENCES products(product_id),

    FOREIGN KEY (seller_id)
        REFERENCES sellers(seller_id)
)
"""

cursor.execute(create_order_items)

In [11]:
# PAYMENTS TABLE
create_payments = """
CREATE TABLE IF NOT EXISTS order_payments (
    order_id VARCHAR(50),
    payment_sequential INT,
    payment_type VARCHAR(30),
    payment_installments INT,
    payment_value DECIMAL(10,2),

    PRIMARY KEY (order_id, payment_sequential),

    FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
)
"""

cursor.execute(create_payments)

In [29]:
# ORDER REVIEWS TABLE
create_reviews = """
CREATE TABLE order_reviews (
    review_id VARCHAR(50),
    order_id VARCHAR(50),
    review_score INT,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date DATETIME,
    review_answer_timestamp DATETIME,

    PRIMARY KEY (review_id, order_id),

    FOREIGN KEY (order_id)
        REFERENCES orders(order_id)
)
"""

cursor.execute(create_reviews)
conn.commit()

In [20]:
conn.commit()

In [21]:
cursor.execute("SHOW TABLES")

for table in cursor.fetchall():
    print(table[0])

customers
geolocation
order_items
order_payments
order_reviews
orders
product_category_translation
products
sellers


In [1]:
import pandas as pd

In [ ]:
customers = pd.read_csv(
    "../data/cleaned/olist_customers_dataset.csv"
)

In [29]:
query = """
INSERT INTO customers (
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
)
VALUES (%s, %s, %s, %s, %s)
"""

data = customers.where(pd.notnull(customers), None).values.tolist()

cursor.executemany(query, data)

conn.commit()

In [12]:
cursor.execute("DELETE FROM order_reviews")
conn.commit()

In [33]:
products = pd.read_csv(
    "../data/cleaned/olist_products_dataset.csv"
)
query = """
INSERT INTO products (
    product_id,
    product_category_name,
    product_name_length,
    product_description_length,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

data = products.where(pd.notnull(products), None).values.tolist()

cursor.executemany(query, data)

conn.commit()

print("Products loaded:", cursor.rowcount)

Products loaded: 32951


In [35]:
sellers = pd.read_csv(
    "../data/cleaned/olist_sellers_dataset.csv"
)
query = """
INSERT INTO sellers (
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
)
VALUES (%s, %s, %s, %s)
"""

data = sellers.where(pd.notnull(sellers), None).values.tolist()

cursor.executemany(query, data)

conn.commit()

print("Sellers loaded:", cursor.rowcount)

Sellers loaded: 3095


In [36]:
translation = pd.read_csv(
    "../data/cleaned/product_category_name_translation.csv"
)
query = """
INSERT INTO product_category_translation (
    product_category_name,
    product_category_name_english
)
VALUES (%s, %s)
"""

data = translation.where(pd.notnull(translation), None).values.tolist()

cursor.executemany(query, data)

conn.commit()

print("Category translations loaded:", cursor.rowcount)

Category translations loaded: 71


In [37]:
orders = pd.read_csv(
    "../data/cleaned/olist_orders_dataset.csv"
)
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [42]:
query = """
INSERT INTO orders (
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

data = orders.where(pd.notnull(orders), None).values.tolist()

cursor.executemany(query, data)

conn.commit()

print("Orders loaded:", cursor.rowcount)

Orders loaded: 99441


In [2]:
order_items = pd.read_csv(
    "../data/cleaned/olist_order_items_dataset.csv"
)

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

In [8]:
query = """
INSERT INTO order_items (
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

data = order_items.where(
    pd.notnull(order_items), None
).values.tolist()

cursor.executemany(query, data)

conn.commit()

print("Order items loaded:", cursor.rowcount)

Order items loaded: 112650


In [9]:
payments = pd.read_csv(
    "../data/cleaned/olist_order_payments_dataset.csv"
)
query = """
INSERT INTO order_payments (
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
)
VALUES (%s, %s, %s, %s, %s)
"""

data = payments.where(pd.notnull(payments), None).values.tolist()

cursor.executemany(query, data)

conn.commit()

print("Payments loaded:", cursor.rowcount)

Payments loaded: 103886


In [10]:
reviews = pd.read_csv(
    "../data/cleaned/olist_order_reviews_dataset.csv"
)
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_columns:
    reviews[col] = pd.to_datetime(
        reviews[col],
        errors="coerce"
    )

In [30]:
query = """
INSERT INTO order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

data = reviews.where(
    pd.notnull(reviews), None
).values.tolist()

cursor.executemany(query, data)

conn.commit()

print("Reviews loaded:", cursor.rowcount)

Reviews loaded: 99224


In [31]:
for chunk in pd.read_csv(
    "../data/cleaned/olist_geolocation_dataset.csv",
    chunksize=10000
):
    data = chunk.where(pd.notnull(chunk), None).values.tolist()

    query = """
    INSERT INTO geolocation (
        geolocation_zip_code_prefix,
        geolocation_lat,
        geolocation_lng,
        geolocation_city,
        geolocation_state
    )
    VALUES (%s, %s, %s, %s, %s)
    """

    cursor.executemany(query, data)
    conn.commit()

print("Geolocation loaded")

Geolocation loaded


In [33]:
tables_sql = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "product_category_translation"
]

for table in tables_sql:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]

    print(table, ":", count)

customers : 99441
geolocation : 1000163
order_items : 112650
order_payments : 103886
order_reviews : 99224
orders : 99441
products : 32951
sellers : 3095
product_category_translation : 71


In [34]:
query = """
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    CONSTRAINT_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'cart2insights'
AND CONSTRAINT_NAME = 'PRIMARY'
ORDER BY TABLE_NAME, ORDINAL_POSITION;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('customers', 'customer_id', 'PRIMARY')
('order_items', 'order_id', 'PRIMARY')
('order_items', 'order_item_id', 'PRIMARY')
('order_payments', 'order_id', 'PRIMARY')
('order_payments', 'payment_sequential', 'PRIMARY')
('order_reviews', 'review_id', 'PRIMARY')
('order_reviews', 'order_id', 'PRIMARY')
('orders', 'order_id', 'PRIMARY')
('product_category_translation', 'product_category_name', 'PRIMARY')
('products', 'product_id', 'PRIMARY')
('sellers', 'seller_id', 'PRIMARY')


In [35]:
query = """
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    CONSTRAINT_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME
FROM information_schema.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'cart2insights'
AND REFERENCED_TABLE_NAME IS NOT NULL
ORDER BY TABLE_NAME, CONSTRAINT_NAME;
"""

cursor.execute(query)

for row in cursor.fetchall():
    print(row)

('order_items', 'order_id', 'order_items_ibfk_1', 'orders', 'order_id')
('order_items', 'product_id', 'order_items_ibfk_2', 'products', 'product_id')
('order_items', 'seller_id', 'order_items_ibfk_3', 'sellers', 'seller_id')
('order_payments', 'order_id', 'order_payments_ibfk_1', 'orders', 'order_id')
('order_reviews', 'order_id', 'order_reviews_ibfk_1', 'orders', 'order_id')
('orders', 'customer_id', 'orders_ibfk_1', 'customers', 'customer_id')
